[![Open In Colab](https://google.com)](https://colab.research.google.com/github/hamagami/IS2026/blob/main/Demo_30_DynaQ_RealAndImagined.ipynb)

# Demo 30: Dyna-Q: Real and Imagined Experience / 実経験と仮想経験

Use each real transition both to update Q-values and to learn a model that provides additional simulated updates. / 実際の遷移からQ値と環境モデルを学び、そのモデルによる仮想更新を加える。

**How to use / 使い方：** Run the cells from top to bottom, then change the controls and compare the figures. / セルを上から順に実行し、操作項目を変えて図を比較する。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass
import ipywidgets as widgets
plt.rcParams.update({'figure.dpi': 105, 'font.size': 11})
np.set_printoptions(precision=3, suppress=True)

In [ ]:
# 共通の5×5迷路と設定 / Shared 5x5 maze and conventions.
# 0行目が上、0列目が左。SからGへの最短経路は8手 / Row 0 is top, column 0 is left; shortest S-to-G path is 8 steps.
N=5
START=(4,0); GOAL=(0,4)
WALLS=frozenset({(1,1),(2,1),(2,3),(3,3),(3,4)})
ACTIONS=[(-1,0),(1,0),(0,-1),(0,1)]
ACTION_NAMES=['Up','Down','Left','Right']
STATES=[(i,j) for i in range(N) for j in range(N) if (i,j) not in WALLS]
NONTERMINAL=[s for s in STATES if s!=GOAL]

def transition(s,a, step_cost=-0.04, goal_reward=1.0, walls=WALLS):
    if s==GOAL:return GOAL,0.0,True
    di,dj=ACTIONS[a];ns=(s[0]+di,s[1]+dj)
    if not(0<=ns[0]<N and 0<=ns[1]<N) or ns in walls:ns=s
    done=ns==GOAL
    return ns, goal_reward if done else step_cost, done

def render_maze(ax,values=None,policy=None,route=None,title='',walls=WALLS,fmt='.2f'):
    from matplotlib.patches import Rectangle
    ax.set_xlim(0,N);ax.set_ylim(N,0);ax.set_aspect('equal');ax.axis('off')
    valid=[v for s,v in (values or {}).items() if s not in walls and np.isfinite(v)]
    lo=min(valid) if valid else 0;hi=max(valid) if valid else 1
    for i in range(N):
        for j in range(N):
            s=(i,j)
            if s in walls:color='#626B7B'
            elif s==GOAL:color='#FAD9CA'
            elif s==START:color='#CDECFD'
            elif values is not None and s in values and hi>lo:
                color=plt.cm.YlGnBu(0.15+0.58*(values[s]-lo)/(hi-lo))
            else:color='#FFFFFF'
            ax.add_patch(Rectangle((j,i),1,1,facecolor=color,edgecolor='#8392A6',linewidth=1.1))
            if s in walls:continue
            if s==GOAL:label='G'
            elif values is not None and s in values:label=f'{values[s]:{fmt}}'
            elif s==START:label='S'
            else:label=''
            if label:ax.text(j+.5,i+.49,label,ha='center',va='center',color='#102E57',fontsize=9 if values is not None else 13,fontweight='bold')
            if policy is not None and s in policy and s!=GOAL and values is None:
                ax.text(j+.5,i+.72,'↑↓←→'[policy[s]],ha='center',va='center',color='#1763A8',fontsize=13)
    if route and len(route)>1:
        xx=[s[1]+.5 for s in route];yy=[s[0]+.5 for s in route]
        ax.plot(xx,yy,lw=3,color='#F39B2F',marker='o',markersize=4,alpha=.92)
    ax.set_title(title,fontweight='bold',fontsize=12)

In [ ]:
def dyna_q(episodes=200,planning=5,seed=0,alpha=.2,gamma=.9,epsilon=.15,cap=100):
    rng=np.random.default_rng(seed);Q={s:np.zeros(4) for s in STATES};model={}
    steps=[];returns=[];success=[];model_sizes=[];total_real=0;total_sim=0
    for ep in range(episodes):
        s=START;score=0;done=False
        for t in range(cap):
            if rng.random()<epsilon:a=int(rng.integers(4))
            else:a=int(rng.choice(np.flatnonzero(np.isclose(Q[s],Q[s].max()))))
            ns,r,done=transition(s,a);score+=r
            target=r+(0 if done else gamma*np.max(Q[ns]));Q[s][a]+=alpha*(target-Q[s][a])
            total_real+=1;model[(s,a)]=(ns,r,done)
            keys=list(model)
            for k in range(planning):
                ss,aa=keys[int(rng.integers(len(keys)))];nn,rr,dd=model[(ss,aa)]
                y=rr+(0 if dd else gamma*np.max(Q[nn]));Q[ss][aa]+=alpha*(y-Q[ss][aa]);total_sim+=1
            s=ns
            if done:break
        steps.append(t+1);returns.append(score);success.append(int(done));model_sizes.append(len(model))
    return Q,np.array(steps),np.array(returns),np.array(success),np.array(model_sizes),total_real,total_sim

for p in [0,5,20]:
    q,steps,ret,win,sz,real,sim=dyna_q(episodes=10,planning=p)
    assert sim==p*real
print('Dyna-Q checks passed. / Dyna-Qの基本確認に成功。')

In [ ]:
episodes=widgets.IntSlider(value=200,min=50,max=500,step=50,description='Episodes / 回数',continuous_update=False)
seed=widgets.IntSlider(value=5,min=0,max=100,description='Seed / 乱数')
repeat=widgets.IntSlider(value=5,min=1,max=12,description='Seeds / 乱数種数',continuous_update=False)
button=widgets.Button(description='Compare / 比較',button_style='primary');out=widgets.Output()
def run(_):
    with out:
        clear_output(wait=True)
        fig,axs=plt.subplots(1,2,figsize=(12,4))
        for planning in (0,5,20):
            arrays=[];wins=[]
            for runid in range(repeat.value):
                q,st,rew,success,m,real,sim=dyna_q(episodes.value,planning=planning,seed=seed.value+runid)
                arrays.append(st);wins.append(success)
            arrays=np.array(arrays);wins=np.array(wins);k=min(25,episodes.value//3)
            avg_steps=arrays.mean(axis=0);avg_wins=wins.mean(axis=0)
            x=np.arange(k,episodes.value+1)
            axs[0].plot(x,np.convolve(avg_steps,np.ones(k)/k,'valid'),label=f'{planning} planning')
            axs[1].plot(x,np.convolve(avg_wins,np.ones(k)/k,'valid'),label=f'{planning} planning')
            print(f'planning={planning:2}: last 30 steps / 直近30回の平均移動数={arrays[:,-30:].mean():.2f}, success={wins[:,-30:].mean():.2f}')
        axs[0].set(title='Real steps / 実環境ステップ数',xlabel='Episode / エピソード',ylabel='Steps / 移動回数')
        axs[1].set(title='Success rate / 成功率',xlabel='Episode / エピソード',ylabel='Success / 成功率',ylim=(0,1))
        for a in axs:a.grid(alpha=.2);a.legend()
        plt.tight_layout();plt.show()
        print('Planning updates use a learned model without new real steps. / 計画更新は学習済みモデルを使い、実環境ステップを増やさない。')
button.on_click(run)
display(widgets.VBox([widgets.HBox([episodes,repeat,seed]),button,out]));run(None)

## One Dyna step / Dynaの1ステップ

1. Act in the real environment to obtain $(s_t,a_t,r_t,s_{t+1})$. / 実環境で行動して $(s_t,a_t,r_t,s_{t+1})$ を取得する。
2. Update the Q-value with real experience (**model-free**). / 実経験でQ値を更新する（**モデルフリー**）。
3. Learn a transition model $(s,a)\to(s',r)$ (**model learning**). / 遷移モデル $(s,a)\to(s',r)$ を学習する（**モデル学習**）。
4. Sample learned state-action transitions to perform additional simulated Q-updates (**planning**). / 学習した状態・行動をモデル内でサンプリングし、追加のQ更新を行う（**計画**）。

This world is deterministic; the model stores the last observed result for each transition. A stochastic world requires a probabilistic model. / この環境は決定的で、各遷移の最後の観測結果をモデルに保存する。確率的環境では確率モデルが必要になる。

## Try it yourself / 実験課題

1. Compare 0, 5, and 20 planning updates under the same number of real interactions. / 実環境での試行回数を揃え、計画更新を0、5、20回に変えて比較する。
2. Explain the difference between real steps and simulated updates. / 実経験とモデル内更新の回数の違いを説明する。
3. Explain how an inaccurate model can make imagined experience harmful. / 不正確なモデルの仮想経験が逆効果になる理由を考える。